##CrewAI Example (Using Ollama for LLMs via Colab)

Overall workflow:
1. Check Python version and install CrewAI
2. Scaffold a CrewAI project (initialized with OpenAI as a placeholder)
3. Install Ollama and start the local inference server
4. Pull the local model (llama3.2:1b)
5. Install LiteLLM support to connect CrewAI to the local model
6. Modify crew.py to switch from OpenAI to Ollama configuration
7. Run the Crew

In [1]:
!python3 --version
# Check python version
# CrewAI requires Python >=3.10 and <3.14, so make sure your environment matches this range.
# We use uv (a fast Python package and environment manager) to automatically handle dependencies and Python versions.

Python 3.12.12


In [2]:
!uv tool install crewai # install crewai

Resolved 131 packages in 1.07s
Prepared 131 packages in 7.39s
Installed 131 packages in 312ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + aiosqlite==0.21.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + appdirs==1.4.4
 + attrs==25.4.0
 + backoff==2.2.1
 + bcrypt==5.0.0
 + build==1.4.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.6
 + chromadb==1.1.1
 + click==8.1.8
 + crewai==1.10.1
 + cryptography==46.0.5
 + deprecation==2.1.0
 + diskcache==5.6.3
 + distro==1.9.0
 + docstring-parser==0.17.0
 + durationpy==0.10
 + et-xmlfile==2.0.0
 + filelock==3.25.2
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + googleapis-common-protos==1.73.0
 + grpcio==1.78.0
 + h11==0.16.0
 + hf-xet==1.4.2
 + httpcore==1.0.9
 + httptools==0.7.1
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + huggingface-hub==1.7.1
 + idna==3.11
 + importlib-metadata==8.7.1
 + importlib-resources==6.5.2
 + instructor==1.14.5
 + jinja2==3.1.6
 + jiter=

In [3]:
!uv tool run crewai create crew latest-ai-development
# This command creates a new CrewAI project with starter code (agents, tasks, and structure).
# We select OpenAI here only to initialize the project. This will be overridden later when we switch to a local LLM (Ollama).
# You may be prompted for an API key depending on the provider selected.
# Since we will use a local model later, this can be skipped or treated as a placeholder.
# If you decide to use an official provider API, always keep your API key private, as it is tied to your usage and billing.

Creating folder latest_ai_development...
Cache expired or not found. Fetching provider data from the web...
Select a provider to set up:
1. openai
2. anthropic
3. gemini
4. nvidia_nim
5. groq
6. huggingface
7. ollama
8. watson
9. bedrock
10. azure
11. cerebras
12. sambanova
13. other
q. Quit
Enter the number of your choice or 'q' to quit: 1
Select a model to use for Openai:
1. gpt-4
2. gpt-4.1
3. gpt-4.1-mini-2025-04-14
4. gpt-4.1-nano-2025-04-14
5. gpt-4o
6. gpt-4o-mini
7. o1-mini
8. o1-preview
q. Quit
Enter the number of your choice or 'q' to quit: 1
Enter your OPENAI API key (press Enter to skip): placeholder
API keys and model saved to .env file
Selected model: gpt-4
  - Created latest_ai_development/.gitignore
  - Created latest_ai_development/pyproject.toml
  - Created latest_ai_development/README.md
  - Created latest_ai_development/knowledge/user_preference.txt
  - Created latest_ai_development/src/latest_ai_development/__init__.py
  - Created latest_ai_development/src/latest_a

Important Setup Note

After running crewai create, a default project structure is generated based on the provider you selected (e.g., OpenAI).

However, in this tutorial, we do not use cloud-based APIs. Instead, we run models locally using Ollama.

Therefore, you need to modify the LLM configuration in crew.py:

Replace the default OpenAI-based setup with a local LLM (Ollama) configuration

A reference version of `crew.py` is provided. Use it to overwrite the default file, or make further revisions as needed.

In [4]:
%cd /content/latest_ai_development

/content/latest_ai_development


In [5]:
!apt-get install -y zstd
# zstd is required to extract the Ollama installation package

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (572 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh
# Install Ollama, which allows us to run LLMs locally on our machine or Colab environment.
# This will serve as the backend model provider instead of using cloud APIs.

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
!nohup ollama serve > ollama.log &
# Start the Ollama inference server in the background, with logs redirected to ollama.log.
# nohup ensures the process keeps running even if the Colab session switches tabs.
# CrewAI will communicate with this server at 127.0.0.1:11434.

nohup: redirecting stderr to stdout


In [8]:
!ollama pull llama3.2:1b
# Download the llama3.2:1b model locally via Ollama.
# The 1b variant is lightweight and well-suited for Colab's free tier.
# If more resources are available, consider larger models for better output quality.
# Once pulled, the model runs entirely locally with no external API calls needed.

In [9]:
!uv add 'crewai[litellm]'
# crewai[litellm] enables LiteLLM support in CrewAI.
# LiteLLM provides a unified interface to connect to multiple LLM providers
# (e.g., OpenAI, Ollama, HuggingFace), making it easier to switch between local and cloud models.
# It is especially useful for non-native or local models (e.g., Ollama).
# For more details, refer to: https://docs.crewai.com/en/learn/llm-connections

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 151 packages in 615ms
Prepared 145 packages in 9.14s
Installed 145 packages in 263ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + aiosqlite==0.21.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + appdirs==1.4.4
 + attrs==25.4.0
 + backoff==2.2.1
 + bcrypt==5.0.0
 + beautifulsoup4==4.13.5
 + build==1.4.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.6
 + chromadb==1.1.1
 + click==8.1.8
 + crewai==1.10.1
 + crewai-tools==1.10.1
 + cryptography==46.0.5
 + defusedxml==0.7.1
 + deprecation==2.1.0
 + diskcache==5.6.3
 + distro==1.9.0
 + docker==7.1.0
 + docstring-parser==0.17.0
 + durationpy==0.10
 + et-xmlfile==2.0.0
 + fastuuid==0.14.0
 + filelock==3.25.2
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + googleapis-common-protos==1.73.0
 + grpcio==1.78.0
 + h11==0.16.0
 + hf-xet==1.4.2
 + httpcore==1.0.9
 +

In [10]:
!uv tool run crewai run # Run the crew

Running the Crew
╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 623b8836-af83-4aac-a2a3-bb74c9844b24                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: research_task                                                         │
│  ID: ebf30